In [2]:
import json
import pandas as pd

REGION_START = 149481000
REGION_END   = 149506500

with open("data/IDS_hg38_region.json") as f:
    region = json.load(f)

seq = region["dna"].upper()

print("chrom          :", region.get("chrom"))
print("start          :", region.get("start"))
print("end            :", region.get("end"))
print("sequence length:", len(seq))
print("expected       :", REGION_END - REGION_START)
print()
print("first 60 bases :", seq[:60])

chrom          : chrX
start          : 149481000
end            : 149506500
sequence length: 25500
expected       : 25500

first 60 bases : GGTTATATGAAGCAACAAAGATACAGTTTCTCTAAGCCAAGTAAAGGTAAAGAACTGTGG


In [4]:
df = pd.read_csv("data/clinvar_download_20261002_085509.tsv", sep="\t", index_col=False)

snv = df[df["Variant type"] == "single nucleotide variant"].copy()

def parse_spdi(s):
    parts = str(s).split(":")
    if len(parts) != 4:
        return None
    acc, pos, ref, alt = parts
    if len(ref) != 1 or len(alt) != 1:
        return None
    return int(pos), ref.upper(), alt.upper()

rows = []
for _, r in snv.iterrows():
    p = parse_spdi(r["Canonical SPDI"])
    if p is None:
        continue
    pos, ref, alt = p
    idx = pos - REGION_START
    base = seq[idx] if 0 <= idx < len(seq) else None
    rows.append({
        "name": r["Name"],
        "pos": pos,
        "ref_clinvar": ref,
        "alt_clinvar": alt,
        "base_genome": base,
        "match": base == ref,
    })

chk = pd.DataFrame(rows)

print("SNVs in file     :", len(snv))
print("with valid SPDI  :", len(chk))
print("within region    :", int(chk["base_genome"].notna().sum()))
print()
print("reference matches:", int(chk["match"].sum()))
print("mismatches       :", int((~chk["match"]).sum()))

bad = chk[~chk["match"]]
if len(bad):
    print()
    print("Mismatching examples:")
    print(bad.head(10).to_string(index=False))

SNVs in file     : 229
with valid SPDI  : 229
within region    : 229

reference matches: 229
mismatches       : 0


In [5]:
COMP = {"A": "T", "C": "G", "G": "C", "T": "A"}

def corrector(ref, alt):
    """Which editor restores ref from alt, and which strand carries the base to edit."""
    # plus strand: alt -> ref
    if alt == "A" and ref == "G":
        return "ABE", "+"
    if alt == "C" and ref == "T":
        return "CBE", "+"
    # minus strand: complement(alt) -> complement(ref)
    if COMP[alt] == "A" and COMP[ref] == "G":
        return "ABE", "-"
    if COMP[alt] == "C" and COMP[ref] == "T":
        return "CBE", "-"
    return None, None

chk[["editor", "edit_strand"]] = chk.apply(
    lambda r: pd.Series(corrector(r["ref_clinvar"], r["alt_clinvar"])), axis=1
)

editable = chk[chk["editor"].notna()].copy()

print("total SNVs            :", len(chk))
print("chemically correctable:", len(editable))
print()
print(editable["editor"].value_counts().to_string())
print()
print("by editor and strand:")
print(editable.groupby(["editor", "edit_strand"]).size().to_string())

total SNVs            : 229
chemically correctable: 85

editor
ABE    54
CBE    31

by editor and strand:
editor  edit_strand
ABE     +              25
        -              29
CBE     +              16
        -              15


In [6]:
def revcomp(s):
    return s.translate(str.maketrans("ACGT", "TGCA"))[::-1]

rc = revcomp(seq)
L  = len(seq)

PROTOSPACER_LEN = 20
WINDOW = (4, 8)   # editable positions within the protospacer, counted from the PAM-distal end

def matches_pam(triplet, pattern):
    if len(triplet) != len(pattern):
        return False
    return all(p == "N" or b == p for b, p in zip(triplet, pattern))

def find_guides(strand_seq, t, pam="NGG"):
    """t = index of the base to edit, in a sequence read 5'->3'."""
    hits = []
    for p in range(WINDOW[0], WINDOW[1] + 1):
        s = t - (p - 1)
        if s < 0 or s + PROTOSPACER_LEN + len(pam) > len(strand_seq):
            continue
        pam_seq = strand_seq[s + PROTOSPACER_LEN : s + PROTOSPACER_LEN + len(pam)]
        if matches_pam(pam_seq, pam):
            hits.append({
                "window_pos": p,
                "protospacer": strand_seq[s : s + PROTOSPACER_LEN],
                "pam": pam_seq,
            })
    return hits

def strand_view(row):
    """Return (sequence read 5'->3' on the edited strand, index of the target base)."""
    i = row["pos"] - REGION_START
    if row["edit_strand"] == "+":
        return seq, i
    return rc, L - 1 - i

# --- check on a single variant ---
ex = editable.iloc[0]
sv, t = strand_view(ex)

print(ex["name"])
print("editor          :", ex["editor"], "| strand:", ex["edit_strand"])
print("reference base  :", ex["ref_clinvar"], "| on edited strand:", sv[t])
print()

guides = find_guides(sv, t)
print("guides found    :", len(guides))
for g in guides:
    ps = g["protospacer"]
    k  = g["window_pos"] - 1
    print(f"  position {g['window_pos']}:  {ps[:k]}[{ps[k]}]{ps[k+1:]}  {g['pam']}")

NM_000202.8(IDS):c.1591C>T (p.Gln531Ter)
editor          : ABE | strand: +
reference base  : G | on edited strand: G

guides found    : 0


In [7]:
rows = []
for idx, r in editable.iterrows():
    sv, t = strand_view(r)
    g = find_guides(sv, t)
    rows.append({"idx": idx, "name": r["name"], "editor": r["editor"],
                 "strand": r["edit_strand"], "n_guides": len(g)})

res = pd.DataFrame(rows).set_index("idx")

print("chemically correctable :", len(res))
print("with at least one guide:", int((res["n_guides"] > 0).sum()))
print("with no guide          :", int((res["n_guides"] == 0).sum()))
print()
print("with a guide, by editor:")
print(res[res["n_guides"] > 0]["editor"].value_counts().to_string())
print()
print("number of guides per variant:")
print(res["n_guides"].value_counts().sort_index().to_string())

ok = res[res["n_guides"] > 0]
if len(ok):
    r = editable.loc[ok.index[0]]
    sv, t = strand_view(r)
    print()
    print("example with a guide:")
    print(r["name"], "|", r["editor"], "| strand", r["edit_strand"])
    for g in find_guides(sv, t):
        ps = g["protospacer"]
        k  = g["window_pos"] - 1
        print(f"  position {g['window_pos']}:  {ps[:k]}[{ps[k]}]{ps[k+1:]}  {g['pam']}")

chemically correctable : 85
with at least one guide: 22
with no guide          : 63

with a guide, by editor:
editor
ABE    13
CBE     9

number of guides per variant:
n_guides
0    63
1    17
2     5

example with a guide:
NM_000202.8(IDS):c.1568A>G (p.Tyr523Cys) | CBE | strand +
  position 6:  CAAAA[T]ACAGTTCCCCTGCA  TGG


In [8]:
IUPAC = {
    "A": "A", "C": "C", "G": "G", "T": "T",
    "R": "AG", "Y": "CT", "N": "ACGT",
}

def matches_pam(observed, pattern):
    if len(observed) != len(pattern):
        return False
    for base, code in zip(observed, pattern):
        if base not in IUPAC[code]:
            return False
    return True


PAMS = [
    ("SpCas9",           "NGG"),
    ("SpG",              "NG"),
    ("SpRY (preferred)", "NRN"),
    ("PAM-less (limit)", "NNN"),
]

summary = []

for label, pam in PAMS:
    n_with = 0
    for idx, r in editable.iterrows():
        sv, t = strand_view(r)
        guides = find_guides(sv, t, pam=pam)
        if len(guides) > 0:
            n_with += 1
    summary.append({
        "variant":    label,
        "PAM":        pam,
        "with_guide": n_with,
        "pct_of_85":  round(100 * n_with / len(editable), 1),
        "pct_of_395": round(100 * n_with / len(df), 1),
    })

summ = pd.DataFrame(summary)
print(summ.to_string(index=False))

         variant PAM  with_guide  pct_of_85  pct_of_395
          SpCas9 NGG          22       25.9         5.6
             SpG  NG          66       77.6        16.7
SpRY (preferred) NRN          79       92.9        20.0
PAM-less (limit) NNN          85      100.0        21.5


In [9]:
def reachable_positions(strand_seq, pam="NGG"):
    """Every index that falls inside the editing window of some valid guide."""
    reachable = set()
    plen = PROTOSPACER_LEN
    for j in range(plen, len(strand_seq) - len(pam) + 1):
        if matches_pam(strand_seq[j:j + len(pam)], pam):
            start = j - plen
            for p in range(WINDOW[0], WINDOW[1] + 1):
                reachable.add(start + p - 1)
    return reachable

reach_plus  = reachable_positions(seq)
reach_minus = reachable_positions(rc)

print("reachable positions, plus strand :", len(reach_plus))
print("reachable positions, minus strand:", len(reach_minus))
print()

n_check = 0
for idx, r in editable.iterrows():
    i = r["pos"] - REGION_START
    if r["edit_strand"] == "+":
        hit = i in reach_plus
    else:
        hit = (L - 1 - i) in reach_minus
    if hit:
        n_check += 1

method_a = int((res["n_guides"] > 0).sum())

print("method A, per-variant scan  :", method_a)
print("method B, whole-region scan :", n_check)
print("agree                       :", method_a == n_check)

reachable positions, plus strand : 6779
reachable positions, minus strand: 5873

method A, per-variant scan  : 22
method B, whole-region scan : 22
agree                       : True


In [10]:
rows = []
for idx, r in editable.iterrows():
    sv, t = strand_view(r)
    guides = find_guides(sv, t, pam="NGG")
    if len(guides) == 0:
        continue

    target_base = "A" if r["editor"] == "ABE" else "C"
    fewest = None

    for g in guides:
        proto = g["protospacer"]
        k = g["window_pos"] - 1
        # the patient carries the variant base at the target position
        patient = proto[:k] + target_base + proto[k + 1:]
        window  = patient[WINDOW[0] - 1 : WINDOW[1]]
        n_bystanders = window.count(target_base) - 1
        if fewest is None or n_bystanders < fewest:
            fewest = n_bystanders

    rows.append({"name": r["name"], "editor": r["editor"], "min_bystanders": fewest})

by = pd.DataFrame(rows)

print("variants with an NGG guide  :", len(by))
print("with a bystander-free guide :", int((by["min_bystanders"] == 0).sum()))
print()
print("fewest bystanders in the window:")
print(by["min_bystanders"].value_counts().sort_index().to_string())

variants with an NGG guide  : 22
with a bystander-free guide : 5

fewest bystanders in the window:
min_bystanders
0     5
1    12
2     3
3     2


In [11]:
def analyse(pam):
    with_guide = 0
    clean      = 0
    for idx, r in editable.iterrows():
        sv, t = strand_view(r)
        guides = find_guides(sv, t, pam=pam)
        if len(guides) == 0:
            continue
        with_guide += 1
        target_base = "A" if r["editor"] == "ABE" else "C"
        for g in guides:
            proto   = g["protospacer"]
            k       = g["window_pos"] - 1
            patient = proto[:k] + target_base + proto[k + 1:]
            window  = patient[WINDOW[0] - 1 : WINDOW[1]]
            if window.count(target_base) == 1:
                clean += 1
                break
    return with_guide, clean


final = []
for label, pam in PAMS:
    w, c = analyse(pam)
    final.append({
        "variant":        label,
        "PAM":            pam,
        "with_guide":     w,
        "pct_any":        round(100 * w / len(df), 1),
        "clean_guide":    c,
        "pct_clean":      round(100 * c / len(df), 1),
    })

final_df = pd.DataFrame(final)
print(final_df.to_string(index=False))
print()
print("total pathogenic variants :", len(df))
print("chemically correctable    :", len(editable))

         variant PAM  with_guide  pct_any  clean_guide  pct_clean
          SpCas9 NGG          22      5.6            5        1.3
             SpG  NG          66     16.7           20        5.1
SpRY (preferred) NRN          79     20.0           34        8.6
PAM-less (limit) NNN          85     21.5           52       13.2

total pathogenic variants : 395
chemically correctable    : 85


In [12]:
WINDOWS = [(4, 8), (4, 7), (4, 6), (5, 7)]

rows = []
for w in WINDOWS:
    WINDOW = w                      # the scanning functions read this
    for label, pam in [("SpCas9", "NGG"), ("PAM-less", "NNN")]:
        n_with, n_clean = analyse(pam)
        rows.append({
            "window":     f"{w[0]}-{w[1]}",
            "width":      w[1] - w[0] + 1,
            "editor":     label,
            "with_guide": n_with,
            "clean":      n_clean,
            "pct_clean":  round(100 * n_clean / len(df), 1),
        })

WINDOW = (4, 8)                     # restore the default
sens = pd.DataFrame(rows)
print(sens.to_string(index=False))

window  width   editor  with_guide  clean  pct_clean
   4-8      5   SpCas9          22      5        1.3
   4-8      5 PAM-less          85     52       13.2
   4-7      4   SpCas9          14      6        1.5
   4-7      4 PAM-less          85     63       15.9
   4-6      3   SpCas9          10      6        1.5
   4-6      3 PAM-less          85     71       18.0
   5-7      3   SpCas9          13      7        1.8
   5-7      3 PAM-less          85     71       18.0


In [13]:
import matplotlib.pyplot as plt

labels = ["Pathogenic\n(ClinVar)", "Chemically\ncorrectable",
          "With an\nNGG guide", "Bystander-free\nguide"]
values = [395, 85, 22, 5]
colors = ["#1565C0", "#2E7D32", "#EF6C00", "#B71C1C"]

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(labels, values, color=colors, edgecolor="black")
for i, v in enumerate(values):
    ax.text(i, v + 8, f"{v}  ({100 * v / 395:.1f}%)", ha="center")
ax.set_ylabel("Pathogenic / likely pathogenic IDS variants")
ax.set_title("From catalogued variant to a clean single-base correction")
ax.set_ylim(0, 450)
plt.tight_layout()
plt.savefig("results/fig4_funnel.png", dpi=200)
plt.close()
print("fig4 written")

fig4 written


In [17]:
import numpy as np

labels  = list(final_df["variant"])
any_v   = list(final_df["with_guide"])
clean_v = list(final_df["clean_guide"])
x = np.arange(len(labels))
w = 0.38

fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(x - w/2, any_v,   w, label="any guide",      color="#EF6C00", edgecolor="black")
ax.bar(x + w/2, clean_v, w, label="bystander-free", color="#2E7D32", edgecolor="black")
for i in range(len(labels)):
    ax.text(x[i] - w/2, any_v[i] + 1.5,   str(any_v[i]),   ha="center", fontsize=9)
    ax.text(x[i] + w/2, clean_v[i] + 1.5, str(clean_v[i]), ha="center", fontsize=9)
ax.axhline(85, color="black", linestyle="--", linewidth=1)
ax.text(-0.45, 87, "chemical ceiling (85)", ha="left", fontsize=9)
ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=9)
ax.set_ylabel("IDS variants")
ax.set_title("What PAM flexibility buys")
ax.set_ylim(0, 100)
ax.legend()
plt.tight_layout()
plt.savefig("results/fig5_pam.png", dpi=200)
plt.close()
print("fig5 written")

fig5 written


In [15]:
sub = sens[sens["editor"] == "PAM-less"]

fig, ax = plt.subplots(figsize=(7, 5))
ax.bar(sub["window"], sub["clean"], color="#2E7D32", edgecolor="black")
for i, v in enumerate(sub["clean"]):
    ax.text(i, v + 1.5, f"{v}  ({100 * v / 395:.1f}%)", ha="center", fontsize=9)
ax.axhline(85, color="black", linestyle="--", linewidth=1)
ax.text(len(sub) - 0.5, 87, "chemical ceiling (85)", ha="right", fontsize=9)
ax.set_xlabel("Editing window (protospacer positions)")
ax.set_ylabel("Variants with a bystander-free guide")
ax.set_title("What a narrower editing window buys (no PAM constraint)")
ax.set_ylim(0, 100)
plt.tight_layout()
plt.savefig("results/fig6_window.png", dpi=200)
plt.close()
print("fig6 written")

fig6 written


In [18]:
WINDOW = (4, 8)

rows = []
for idx, r in editable.iterrows():
    sv, t = strand_view(r)
    guides = find_guides(sv, t, pam="NGG")
    if len(guides) == 0:
        continue

    target_base = "A" if r["editor"] == "ABE" else "C"

    for g in guides:
        proto   = g["protospacer"]
        k       = g["window_pos"] - 1
        patient = proto[:k] + target_base + proto[k + 1:]
        window  = patient[WINDOW[0] - 1 : WINDOW[1]]
        n_by    = window.count(target_base) - 1

        rows.append({
            "variant":         r["name"],
            "editor":          r["editor"],
            "edited_strand":   r["edit_strand"],
            "grch38_position": r["pos"] + 1,
            "reference":       r["ref_clinvar"],
            "variant_allele":  r["alt_clinvar"],
            "protospacer":     proto,
            "pam":             g["pam"],
            "window_position": g["window_pos"],
            "bystanders":      n_by,
            "clean":           "yes" if n_by == 0 else "no",
        })

guides_df = pd.DataFrame(rows).sort_values(["clean", "variant"], ascending=[False, True])
guides_df.to_csv("results/guides_spcas9.csv", index=False)

print("guides written  :", len(guides_df))
print("variants covered:", guides_df["variant"].nunique())
print("clean guides    :", int((guides_df["clean"] == "yes").sum()))

guides written  : 27
variants covered: 22
clean guides    : 5
